In [1]:
import findspark
findspark.init("/usr/local/spark")
from pyspark.sql import SparkSession, functions as F

In [2]:
spark = SparkSession.builder \
    .appName("Spark YARN Example") \
    .master("yarn") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .enableHiveSupport() \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2059e1a9-5ae0-4be5-8ad7-a5f268712ee3;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.6.0/postgresql-42.6.0.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.6.0!postgresql.jar (531ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.31.0/checker-qual-3.31.0.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.31.0!checker-qual.jar (197ms)
:: resolution report :: resolve 1154ms :: artifacts dl 734ms
	:: modules in use:
	org.checkerframework#checker-qual;3.31.0 from central in [default]
	org.postgresql#postgresql;42.6.0 from central in [default]
	-------------------------------------------

In [3]:
! curl -O https://raw.githubusercontent.com/erkansirin78/datasets/master/Advertising.csv
! hdfs dfs -put Advertising.csv /user/root/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4556  100  4556    0     0  10284      0 --:--:-- --:--:-- --:--:-- 10284


In [4]:
df_hdfs = spark.read.format("csv").option("header", True) \
.option("inferSchema", True).option("sep",",") \
.load("hdfs://cluster-master:9000/user/root/Advertising.csv")

In [5]:
df_hdfs.show(5)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
|  5|180.8| 10.8|     58.4| 12.9|
+---+-----+-----+---------+-----+
only showing top 5 rows



In [6]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [7]:
spark.sql("use default")

DataFrame[]

In [8]:
spark.sql("show tables;").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [9]:
df_hdfs.write.mode("overwrite").format("orc").saveAsTable("default.spark_adv")

In [10]:
spark.sql("show tables;").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|spark_adv|      false|
+---------+---------+-----------+



In [11]:
df_hive = spark.table("default.spark_adv")

In [12]:
df_hive.show(4)

+---+-----+-----+---------+-----+
| ID|   TV|Radio|Newspaper|Sales|
+---+-----+-----+---------+-----+
|  1|230.1| 37.8|     69.2| 22.1|
|  2| 44.5| 39.3|     45.1| 10.4|
|  3| 17.2| 45.9|     69.3|  9.3|
|  4|151.5| 41.3|     58.5| 18.5|
+---+-----+-----+---------+-----+
only showing top 4 rows

